In [1]:
import pandas as pd
from pathlib import Path

In [2]:
dossier_dvf = Path("../../data/extracted/dvf")

fichiers = sorted(
    dossier_dvf.glob("*.txt")
)

fichiers

[WindowsPath('../../data/extracted/dvf/ValeursFoncieres-2021.txt'),
 WindowsPath('../../data/extracted/dvf/ValeursFoncieres-2022.txt'),
 WindowsPath('../../data/extracted/dvf/ValeursFoncieres-2023.txt'),
 WindowsPath('../../data/extracted/dvf/ValeursFoncieres-2024.txt'),
 WindowsPath('../../data/extracted/dvf/ValeursFoncieres-2025.txt')]

In [3]:
import pandas as pd

dfs = []

for fichier in fichiers:
    print("Lecture :", fichier.name)

    df_temp = pd.read_csv(
        fichier,
        sep="|",
        low_memory=False
    )

    print(df_temp.shape)
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

print("Dataset final :", df.shape)

Lecture : ValeursFoncieres-2021.txt
(4674542, 43)
Lecture : ValeursFoncieres-2022.txt
(4676187, 43)
Lecture : ValeursFoncieres-2023.txt
(3817426, 43)
Lecture : ValeursFoncieres-2024.txt
(3499931, 43)
Lecture : ValeursFoncieres-2025.txt
(3714829, 43)
Dataset final : (20382915, 43)


In [4]:
df["Date mutation"] = pd.to_datetime(
    df["Date mutation"],
    dayfirst=True,
    errors="coerce"
)

df["Date mutation"].dt.year.value_counts().sort_index()

Date mutation
2021    4674542
2022    4676187
2023    3817426
2024    3499931
2025    3714829
Name: count, dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20382915 entries, 0 to 20382914
Data columns (total 43 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   Identifiant de document     float64       
 1   Reference document          float64       
 2   1 Articles CGI              float64       
 3   2 Articles CGI              float64       
 4   3 Articles CGI              float64       
 5   4 Articles CGI              float64       
 6   5 Articles CGI              float64       
 7   No disposition              int64         
 8   Date mutation               datetime64[ns]
 9   Nature mutation             object        
 10  Valeur fonciere             object        
 11  No voie                     float64       
 12  B/T/Q                       object        
 13  Type de voie                object        
 14  Code voie                   object        
 15  Voie                        object        
 16  Code postal     

In [6]:
df.isnull().sum().sort_values(
    ascending=False
).head(20)

Identifiant de document       20382915
Reference document            20382915
1 Articles CGI                20382915
2 Articles CGI                20382915
3 Articles CGI                20382915
4 Articles CGI                20382915
5 Articles CGI                20382915
Identifiant local             20382915
Surface Carrez du 5eme lot    20377055
Surface Carrez du 4eme lot    20366397
No Volume                     20336924
5eme lot                      20333459
Surface Carrez du 3eme lot    20315867
4eme lot                      20270998
3eme lot                      20036784
Surface Carrez du 2eme lot    19784857
Nature culture speciale       19505538
B/T/Q                         19460743
Prefixe de section            19417281
Surface Carrez du 1er lot     18553498
dtype: int64

In [7]:
df["Nature mutation"].value_counts()

Nature mutation
Vente                                 18982926
Vente en l'état futur d'achèvement     1046189
Echange                                 237573
Vente terrain à bâtir                    54555
Adjudication                             51289
Expropriation                            10383
Name: count, dtype: int64

In [8]:
df["Type local"].value_counts()

Type local
Dépendance                                  5308320
Maison                                      3355671
Appartement                                 2879521
Local industriel. commercial ou assimilé     650255
Name: count, dtype: int64

In [9]:
df_filtre = df[
    (df["Nature mutation"] == "Vente")
    &
    (
        df["Type local"].isin(
            ["Maison", "Appartement"]
        )
    )
]

In [10]:
df_filtre.shape

(6080977, 43)

In [13]:
colonnes_importantes = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Code postal",
    "Commune",
    "Code departement",
    "Code commune",
    "Type local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Surface terrain"
]

df_filtre[
    colonnes_importantes
].isna().sum()

Date mutation                      0
Nature mutation                    0
Valeur fonciere                32575
Code postal                      249
Commune                            0
Code departement                   0
Code commune                       0
Type local                         0
Surface reelle bati              542
Nombre pieces principales        542
Surface terrain              2250767
dtype: int64

In [14]:
df_filtre.groupby(
    "Type local"
)["Surface terrain"].apply(
    lambda x: x.isnull().sum()
)

Type local
Appartement    2111412
Maison          139355
Name: Surface terrain, dtype: int64

In [15]:
df_clean = df_filtre.copy()

df_clean = df_clean.dropna(
    subset=["Valeur fonciere"]
)

In [16]:
df_clean.dtypes

Identifiant de document              float64
Reference document                   float64
1 Articles CGI                       float64
2 Articles CGI                       float64
3 Articles CGI                       float64
4 Articles CGI                       float64
5 Articles CGI                       float64
No disposition                         int64
Date mutation                 datetime64[ns]
Nature mutation                       object
Valeur fonciere                       object
No voie                              float64
B/T/Q                                 object
Type de voie                          object
Code voie                             object
Voie                                  object
Code postal                          float64
Commune                               object
Code departement                      object
Code commune                           int64
Prefixe de section                   float64
Section                               object
No plan   

In [17]:
df_clean["Valeur fonciere"].describe()

count       6048402
unique       265079
top       150000,00
freq          61197
Name: Valeur fonciere, dtype: object

In [18]:
df_clean["Surface reelle bati"].describe()

count    6.047861e+06
mean     8.200208e+01
std      4.667668e+01
min      0.000000e+00
25%      5.000000e+01
50%      7.500000e+01
75%      1.020000e+02
max      3.160000e+03
Name: Surface reelle bati, dtype: float64

In [19]:
df_clean["Surface terrain"].describe()

count    3.801380e+06
mean     1.514789e+03
std      1.305316e+04
min      0.000000e+00
25%      2.600000e+02
50%      5.000000e+02
75%      9.120000e+02
max      3.760000e+06
Name: Surface terrain, dtype: float64

In [20]:
colonnes_utiles = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Code postal",
    "Commune",
    "Code departement",
    "Code commune",
    "Type local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Surface terrain"
]

df_clean = df_clean[colonnes_utiles]

In [25]:
df_clean.isnull().sum()

Date mutation                      0
Nature mutation                    0
Valeur fonciere                    0
Code postal                      244
Commune                            0
Code departement                   0
Code commune                       0
Type local                         0
Surface reelle bati              541
Nombre pieces principales        541
Surface terrain              2247022
dtype: int64

In [26]:
df_clean.shape

(6048402, 11)

In [27]:
df_clean["Type local"].value_counts()

Type local
Maison         3327400
Appartement    2721002
Name: count, dtype: int64

In [28]:
df_clean["Nature mutation"].value_counts()

Nature mutation
Vente    6048402
Name: count, dtype: int64

In [29]:
df_clean["Valeur fonciere"].head()

1     185000,00
3     204332,00
5     320000,00
7     176000,00
14    226700,00
Name: Valeur fonciere, dtype: object

In [30]:
df_clean["Valeur fonciere"] = (
    df_clean["Valeur fonciere"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

In [31]:
df_clean = df_clean.dropna(
    subset=["Code postal"]
)

In [32]:
df_clean = df_clean.dropna(
    subset=["Surface reelle bati"]
)

In [33]:
df_clean = df_clean.dropna(
    subset=["Nombre pieces principales"]
)

In [34]:
df_clean.groupby(
    "Type local"
)["Surface terrain"].apply(
    lambda x: x.isna().sum()
)

Type local
Appartement    2107550
Maison          139126
Name: Surface terrain, dtype: int64

In [35]:
df_clean.isnull().sum()

Date mutation                      0
Nature mutation                    0
Valeur fonciere                    0
Code postal                        0
Commune                            0
Code departement                   0
Code commune                       0
Type local                         0
Surface reelle bati                0
Nombre pieces principales          0
Surface terrain              2246676
dtype: int64

In [36]:
df_clean["Type local"].value_counts()

Type local
Maison         3326999
Appartement    2720618
Name: count, dtype: int64

In [37]:
df_clean["Valeur fonciere"].describe()

count    6.047617e+06
mean     2.004065e+06
std      1.956393e+07
min      1.500000e-01
25%      1.200000e+05
50%      2.000000e+05
75%      3.350000e+05
max      7.225900e+08
Name: Valeur fonciere, dtype: float64

In [38]:
df_clean["Surface reelle bati"].describe()

count    6.047617e+06
mean     8.200180e+01
std      4.667672e+01
min      0.000000e+00
25%      5.000000e+01
50%      7.500000e+01
75%      1.020000e+02
max      3.160000e+03
Name: Surface reelle bati, dtype: float64

In [39]:
df_clean["Nombre pieces principales"].describe()

count    6.047617e+06
mean     3.464305e+00
std      1.614617e+00
min      0.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      4.000000e+00
max      1.980000e+02
Name: Nombre pieces principales, dtype: float64

In [40]:
df_clean["Surface terrain"].describe()

count    3.800941e+06
mean     1.514820e+03
std      1.305389e+04
min      0.000000e+00
25%      2.600000e+02
50%      5.000000e+02
75%      9.120000e+02
max      3.760000e+06
Name: Surface terrain, dtype: float64

In [41]:
(df_clean["Valeur fonciere"] <= 1000).sum()

np.int64(23314)

In [42]:
(df_clean["Surface reelle bati"] <= 5).sum()

np.int64(2569)

In [43]:
(df_clean["Nombre pieces principales"] == 0).sum()

np.int64(11790)

In [44]:
df_clean[
    df_clean["Valeur fonciere"] <= 1000
].head(20)

,Date mutation,Nature mutation,Valeur fonciere,Code postal,Commune,Code departement,Code commune,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain
4875,2021-05-05,Vente,700.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,27.0,2.0,NaN
4876,2021-05-05,Vente,700.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,55.0,3.0,NaN
4879,2021-05-05,Vente,700.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,88.0,3.0,NaN
8205,2021-07-19,Vente,1.0,1250.0,JOURNANS,01,197,Maison,99.0,4.0,543.0
8206,2021-07-19,Vente,1.0,1250.0,JOURNANS,01,197,Appartement,80.0,4.0,568.0
8207,2021-07-19,Vente,1.0,1250.0,JOURNANS,01,197,Appartement,79.0,4.0,568.0
8208,2021-07-19,Vente,1.0,1250.0,JOURNANS,01,197,Appartement,65.0,3.0,568.0
8209,2021-07-19,Vente,1.0,1250.0,JOURNANS,01,197,Appartement,50.0,2.0,568.0
8210,2021-07-19,Vente,1.0,1250.0,JOURNANS,01,197,Appartement,53.0,2.0,568.0
8211,2021-07-19,Vente,1.0,1250.0,JOURNANS,01,197,Appartement,65.0,3.0,568.0


In [45]:
df_clean[
    df_clean["Surface reelle bati"] <= 5
].head(20)

,Date mutation,Nature mutation,Valeur fonciere,Code postal,Commune,Code departement,Code commune,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain
3997,2021-04-29,Vente,315000.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,4.0,0.0,NaN
11064,2021-10-04,Vente,250000.0,1500.0,AMBERIEU-EN-BUGEY,01,4,Appartement,4.0,0.0,NaN
17655,2021-11-19,Vente,86000.0,1130.0,NANTUA,01,269,Appartement,1.0,0.0,NaN
20042,2021-11-16,Vente,140000.0,1700.0,SAINT-MAURICE-DE-BEYNOST,01,376,Appartement,1.0,0.0,NaN
26121,2021-12-22,Vente,102000.0,1700.0,SAINT-MAURICE-DE-BEYNOST,01,376,Appartement,1.0,0.0,NaN
37499,2021-06-04,Vente,645000.0,1630.0,SERGY,01,401,Appartement,5.0,0.0,NaN
43257,2021-08-31,Vente,147000.0,1210.0,FERNEY-VOLTAIRE,01,160,Appartement,3.0,0.0,NaN
43947,2021-08-30,Vente,62000.0,1170.0,GEX,01,173,Appartement,2.0,0.0,NaN
44971,2021-09-20,Vente,180000.0,1210.0,FERNEY-VOLTAIRE,01,160,Appartement,2.0,0.0,NaN
45562,2021-01-09,Vente,122200.0,1700.0,SAINT-MAURICE-DE-BEYNOST,01,376,Appartement,1.0,0.0,NaN


In [46]:
df_clean[
    df_clean["Nombre pieces principales"] == 0
].head(20)

,Date mutation,Nature mutation,Valeur fonciere,Code postal,Commune,Code departement,Code commune,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain
240,2021-01-06,Vente,31500.0,1750.0,CROTTET,01,134,Appartement,7.0,0.0,NaN
250,2021-01-11,Vente,439010.0,1250.0,CEYZERIAT,01,72,Maison,17.0,0.0,1225.0
251,2021-01-11,Vente,439010.0,1250.0,CEYZERIAT,01,72,Maison,17.0,0.0,775.0
253,2021-01-11,Vente,439010.0,1250.0,CEYZERIAT,01,72,Maison,17.0,0.0,1335.0
3997,2021-04-29,Vente,315000.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,4.0,0.0,NaN
5288,2021-05-25,Vente,974000.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,28.0,0.0,70.0
5289,2021-05-25,Vente,974000.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,25.0,0.0,70.0
8225,2021-07-19,Vente,45000.0,1960.0,PERONNAS,01,289,Appartement,20.0,0.0,NaN
10345,2021-09-10,Vente,114000.0,1290.0,PONT-DE-VEYLE,01,306,Maison,29.0,0.0,400.0
11064,2021-10-04,Vente,250000.0,1500.0,AMBERIEU-EN-BUGEY,01,4,Appartement,4.0,0.0,NaN


In [47]:
df_clean = df_clean[
    df_clean["Valeur fonciere"] > 1000
]

In [48]:
df_clean = df_clean[
    df_clean["Surface reelle bati"] > 5
]

In [49]:
len(df_clean)

6021766

In [50]:
df_clean.isnull().sum()

Date mutation                      0
Nature mutation                    0
Valeur fonciere                    0
Code postal                        0
Commune                            0
Code departement                   0
Code commune                       0
Type local                         0
Surface reelle bati                0
Nombre pieces principales          0
Surface terrain              2238525
dtype: int64

In [51]:
df_clean["prix_m2"] = (
    df_clean["Valeur fonciere"]
    / df_clean["Surface reelle bati"]
)

In [52]:
df_clean["prix_m2"].describe()

count    6.021766e+06
mean     3.545360e+04
std      3.835732e+05
min      3.960396e+00
25%      1.695652e+03
50%      2.710833e+03
75%      4.565217e+03
max      4.663156e+07
Name: prix_m2, dtype: float64

In [53]:
df_clean.nlargest(
    20,
    "prix_m2"
)[[
    "Date mutation",
    "Commune",
    "Type local",
    "Valeur fonciere",
    "Surface reelle bati",
    "prix_m2"
]]

,Date mutation,Commune,Type local,Valeur fonciere,Surface reelle bati,prix_m2
9278436,2022-07-28,PARIS 08,Appartement,606210300.0,13.0,4.663156e+07
20376958,2025-11-19,PARIS 16,Appartement,695000000.0,22.0,3.159091e+07
9278373,2022-07-28,PARIS 08,Appartement,606210300.0,22.0,2.755501e+07
20376990,2025-11-19,PARIS 16,Appartement,695000000.0,26.0,2.673077e+07
7286381,2022-12-27,VANNES,Appartement,314985152.0,12.0,2.624876e+07
7286396,2022-12-27,VANNES,Appartement,314985152.0,12.0,2.624876e+07
7286406,2022-12-27,VANNES,Appartement,314985152.0,12.0,2.624876e+07
7286407,2022-12-27,VANNES,Appartement,314985152.0,12.0,2.624876e+07
7286412,2022-12-27,VANNES,Appartement,314985152.0,12.0,2.624876e+07
7286414,2022-12-27,VANNES,Appartement,314985152.0,12.0,2.624876e+07


In [54]:
df_clean.nsmallest(
    20,
    "prix_m2"
)[[
    "Date mutation",
    "Commune",
    "Type local",
    "Valeur fonciere",
    "Surface reelle bati",
    "prix_m2"
]]

,Date mutation,Commune,Type local,Valeur fonciere,Surface reelle bati,prix_m2
10146695,2023-03-01,BOURROU,Maison,2000.0,505.0,3.960396
2376735,2021-06-14,TERRANJOU,Maison,1750.0,400.0,4.375000
2376738,2021-06-14,TERRANJOU,Maison,1750.0,400.0,4.375000
9980224,2023-02-27,COLOMBIERS,Maison,1200.0,260.0,4.615385
2243863,2021-09-16,SAINT-HILAIRE-SAINT-MESMIN,Maison,1040.0,217.0,4.792627
8482939,2022-08-08,TREVIEN,Maison,1200.0,250.0,4.800000
7811668,2022-05-09,MONTAGNY,Maison,1050.0,214.0,4.906542
7811677,2022-05-09,MONTAGNY,Maison,1050.0,214.0,4.906542
7811682,2022-05-09,MONTAGNY,Maison,1050.0,214.0,4.906542
7016485,2022-08-05,SAUMUR,Maison,1500.0,300.0,5.000000


In [55]:
Q1 = df_clean["prix_m2"].quantile(0.25)
Q3 = df_clean["prix_m2"].quantile(0.75)

IQR = Q3 - Q1

print("Q1 :", Q1)
print("Q3 :", Q3)
print("IQR :", IQR)

Q1 : 1695.6521739130435
Q3 : 4565.217391304348
IQR : 2869.5652173913045


In [56]:
borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

print("Borne inf :", borne_inf)
print("Borne sup :", borne_sup)

Borne inf : -2608.6956521739135
Borne sup : 8869.565217391304


In [57]:
outliers_prix_m2 = (
    (df_clean["prix_m2"] > borne_sup)
).sum()

print(outliers_prix_m2)

653856


In [58]:
outliers_pct = (
    outliers_prix_m2 / len(df_clean)
) * 100

print(outliers_pct)

10.858210033402163


In [59]:
df_clean[
    df_clean["prix_m2"] > borne_sup
]["Commune"].value_counts().head(20)

Commune
PARIS 15                14577
PARIS 16                12723
PARIS 17                12641
PARIS 18                12426
PARIS 11                12337
PARIS 14                 7281
PARIS 12                 7064
PARIS 10                 6854
TOULOUSE                 6651
PARIS 20                 6534
NICE                     6263
PARIS 09                 6076
VANNES                   5665
LORIENT                  5631
PARIS 13                 5625
PARIS 07                 5452
PARIS 19                 5306
AUXERRE                  5212
BORDEAUX                 5207
BOULOGNE-BILLANCOURT     5192
Name: count, dtype: int64

In [61]:
df_clean["prix_m2"].quantile([
    0.90,
    0.95,
    0.99,
    0.995,
    0.999
])

0.900    9.510000e+03
0.950    1.990000e+04
0.990    5.172414e+05
0.995    1.839965e+06
0.999    5.962791e+06
Name: prix_m2, dtype: float64

In [62]:
df_clean["Valeur fonciere"].quantile([
    0.90,
    0.95,
    0.99,
    0.995,
    0.999
])

0.900       625000.0
0.950      1219200.0
0.990     27211160.0
0.995    103038040.0
0.999    337200416.0
Name: Valeur fonciere, dtype: float64

In [63]:
df_clean.nlargest(
    20,
    "Valeur fonciere"
)[[
    "Date mutation",
    "Commune",
    "Type local",
    "Valeur fonciere",
    "Surface reelle bati",
    "prix_m2"
]]

,Date mutation,Commune,Type local,Valeur fonciere,Surface reelle bati,prix_m2
7419183,2022-10-28,MARCQ EN BAROEUL,Maison,722590020.0,119.0,6.072185e+06
20376958,2025-11-19,PARIS 16,Appartement,695000000.0,22.0,3.159091e+07
20376959,2025-11-19,PARIS 16,Appartement,695000000.0,74.0,9.391892e+06
20376960,2025-11-19,PARIS 16,Appartement,695000000.0,90.0,7.722222e+06
20376961,2025-11-19,PARIS 16,Appartement,695000000.0,90.0,7.722222e+06
20376962,2025-11-19,PARIS 16,Appartement,695000000.0,90.0,7.722222e+06
20376963,2025-11-19,PARIS 16,Appartement,695000000.0,91.0,7.637363e+06
20376964,2025-11-19,PARIS 16,Appartement,695000000.0,109.0,6.376147e+06
20376968,2025-11-19,PARIS 16,Appartement,695000000.0,86.0,8.081395e+06
20376969,2025-11-19,PARIS 16,Appartement,695000000.0,110.0,6.318182e+06


In [64]:
df_clean[
    df_clean["Valeur fonciere"] >= 100_000_000
].shape

(31568, 12)

In [65]:
(
    df_clean["Valeur fonciere"] >= 100_000_000
).sum()

np.int64(31568)

In [66]:
(
    (
        df_clean["Valeur fonciere"] >= 100_000_000
    ).sum()
    / len(df_clean)
) * 100

np.float64(0.524231595847464)

In [67]:
df_clean = df_clean[
    df_clean["Valeur fonciere"] < 100_000_000
].copy()

In [68]:
df_clean["prix_m2"] = (
    df_clean["Valeur fonciere"]
    / df_clean["Surface reelle bati"]
)

In [69]:
df_clean["prix_m2"].describe()

count    5.990198e+06
mean     1.209133e+04
std      8.302170e+04
min      3.960396e+00
25%      1.690789e+03
50%      2.698413e+03
75%      4.516129e+03
max      1.335833e+07
Name: prix_m2, dtype: float64

In [70]:
df_clean["prix_m2"].quantile([
    0.90,
    0.95,
    0.99,
    0.995,
    0.999
])

0.900    9.150000e+03
0.950    1.714286e+04
0.990    2.100000e+05
0.995    4.910609e+05
0.999    1.168946e+06
Name: prix_m2, dtype: float64

In [71]:
df_clean.nlargest(
    20,
    "prix_m2"
)[[
    "Date mutation",
    "Commune",
    "Type local",
    "Valeur fonciere",
    "Surface reelle bati",
    "Nombre pieces principales",
    "prix_m2"
]]

,Date mutation,Commune,Type local,Valeur fonciere,Surface reelle bati,Nombre pieces principales,prix_m2
16639246,2024-04-23,PARIS 06,Appartement,80150000.0,6.0,1.0,1.335833e+07
9281255,2022-07-26,PARIS 08,Appartement,57500000.0,6.0,1.0,9.583333e+06
16639227,2024-04-23,PARIS 06,Appartement,80150000.0,10.0,1.0,8.015000e+06
4650670,2021-10-11,PARIS 16,Appartement,78000000.0,11.0,1.0,7.090909e+06
4598901,2021-05-31,PARIS 08,Appartement,56000000.0,8.0,1.0,7.000000e+06
13167137,2023-12-28,PARIS 13,Appartement,91000000.0,13.0,1.0,7.000000e+06
13167493,2023-12-28,PARIS 13,Appartement,91000000.0,13.0,2.0,7.000000e+06
13167503,2023-12-28,PARIS 13,Appartement,91000000.0,13.0,1.0,7.000000e+06
16667250,2024-12-19,PARIS 16,Appartement,68820000.0,10.0,1.0,6.882000e+06
13167077,2023-12-28,PARIS 13,Appartement,91000000.0,14.0,1.0,6.500000e+06


In [72]:
(
    df_clean["prix_m2"] > 50_000
).sum()

np.int64(159070)

In [73]:
(
    (df_clean["prix_m2"] > 50_000).sum()
    / len(df_clean)
) * 100

np.float64(2.655504876466521)

In [74]:
(df_clean["Surface reelle bati"] <= 15).sum()

np.int64(57247)

In [75]:
(
    (df_clean["Surface reelle bati"] <= 15).sum()
    / len(df_clean)
) * 100

np.float64(0.9556779258381776)

In [76]:
df_clean["Surface reelle bati"].quantile([
    0.01,
    0.05,
    0.10,
    0.25,
    0.50
])

0.01    16.0
0.05    24.0
0.10    32.0
0.25    50.0
0.50    75.0
Name: Surface reelle bati, dtype: float64

In [77]:
df_temp = df_clean[
    df_clean["Surface reelle bati"] >= 15
].copy()

df_temp["prix_m2"] = (
    df_temp["Valeur fonciere"]
    / df_temp["Surface reelle bati"]
)

df_temp["prix_m2"].describe()

count    5.948654e+06
mean     1.172409e+04
std      7.923911e+04
min      3.960396e+00
25%      1.685833e+03
50%      2.684211e+03
75%      4.467391e+03
max      5.750000e+06
Name: prix_m2, dtype: float64

In [78]:
df_temp["prix_m2"].quantile([
    0.90,
    0.95,
    0.99,
    0.995,
    0.999
])

0.900    8.879484e+03
0.950    1.612162e+04
0.990    2.024792e+05
0.995    4.772727e+05
0.999    1.153775e+06
Name: prix_m2, dtype: float64

In [79]:
df_temp[
    df_temp["prix_m2"] > 200_000
][[
    "Date mutation",
    "Commune",
    "Type local",
    "Valeur fonciere",
    "Surface reelle bati",
    "Nombre pieces principales",
    "prix_m2"
]].head(30)

,Date mutation,Commune,Type local,Valeur fonciere,Surface reelle bati,Nombre pieces principales,prix_m2
19578,2021-11-30,BELIGNEUX,Appartement,20037016.0,36.0,2.0,556583.777778
19580,2021-11-30,BELIGNEUX,Appartement,20037016.0,36.0,2.0,556583.777778
19581,2021-11-30,BELIGNEUX,Appartement,20037016.0,36.0,2.0,556583.777778
19582,2021-11-30,BELIGNEUX,Appartement,20037016.0,36.0,2.0,556583.777778
19583,2021-11-30,BELIGNEUX,Appartement,20037016.0,36.0,2.0,556583.777778
19584,2021-11-30,BELIGNEUX,Appartement,20037016.0,36.0,2.0,556583.777778
19585,2021-11-30,BELIGNEUX,Appartement,20037016.0,36.0,2.0,556583.777778
19586,2021-11-30,BELIGNEUX,Appartement,20037016.0,36.0,2.0,556583.777778
19587,2021-11-30,BELIGNEUX,Appartement,20037016.0,36.0,2.0,556583.777778
19588,2021-11-30,BELIGNEUX,Appartement,20037016.0,36.0,2.0,556583.777778


In [80]:
(
    df_temp["prix_m2"] > 200_000
).sum()

np.int64(60102)

In [81]:
(
    (df_temp["prix_m2"] > 200_000).sum()
    / len(df_temp)
) * 100

np.float64(1.0103462060493011)

In [82]:
df_clean.duplicated().sum()

np.int64(277963)

In [83]:
(
    df_clean.duplicated().sum()
    / len(df_clean)
) * 100

np.float64(4.64029736579659)

In [84]:
df_clean[
    df_clean.duplicated(keep=False)
].head(20)

,Date mutation,Nature mutation,Valeur fonciere,Code postal,Commune,Code departement,Code commune,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain,prix_m2
55,2021-01-07,Vente,72000.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,61.0,3.0,NaN,1180.327869
58,2021-01-07,Vente,72000.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,61.0,3.0,NaN,1180.327869
337,2021-01-08,Vente,201400.0,1370.0,BENY,01,38,Maison,104.0,5.0,614.0,1936.538462
338,2021-01-08,Vente,201400.0,1370.0,BENY,01,38,Maison,104.0,5.0,614.0,1936.538462
365,2021-01-15,Vente,313550.0,1160.0,SAINT-MARTIN-DU-MONT,01,374,Maison,154.0,6.0,1585.0,2036.038961
366,2021-01-15,Vente,313550.0,1160.0,SAINT-MARTIN-DU-MONT,01,374,Maison,154.0,6.0,1585.0,2036.038961
517,2021-01-25,Vente,87500.0,1750.0,SAINT-LAURENT-SUR-SAONE,01,370,Appartement,46.0,2.0,NaN,1902.173913
518,2021-01-25,Vente,87500.0,1750.0,SAINT-LAURENT-SUR-SAONE,01,370,Appartement,46.0,2.0,NaN,1902.173913
750,2021-01-21,Vente,88000.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,18.0,1.0,NaN,4888.888889
751,2021-01-21,Vente,88000.0,1000.0,BOURG-EN-BRESSE,01,53,Appartement,18.0,1.0,NaN,4888.888889


In [85]:
beligneux = df_clean[
    (df_clean["Commune"] == "BELIGNEUX")
    &
    (df_clean["Valeur fonciere"] == 20037016)
]

beligneux.shape

(32, 12)

In [86]:
beligneux.head(20)

,Date mutation,Nature mutation,Valeur fonciere,Code postal,Commune,Code departement,Code commune,Type local,Surface reelle bati,Nombre pieces principales,Surface terrain,prix_m2
19578,2021-11-30,Vente,20037016.0,1360.0,BELIGNEUX,01,32,Appartement,36.0,2.0,3000.0,556583.777778
19580,2021-11-30,Vente,20037016.0,1360.0,BELIGNEUX,01,32,Appartement,36.0,2.0,3000.0,556583.777778
19581,2021-11-30,Vente,20037016.0,1360.0,BELIGNEUX,01,32,Appartement,36.0,2.0,3000.0,556583.777778
19582,2021-11-30,Vente,20037016.0,1360.0,BELIGNEUX,01,32,Appartement,36.0,2.0,3000.0,556583.777778
19583,2021-11-30,Vente,20037016.0,1360.0,BELIGNEUX,01,32,Appartement,36.0,2.0,3000.0,556583.777778
19584,2021-11-30,Vente,20037016.0,1360.0,BELIGNEUX,01,32,Appartement,36.0,2.0,3000.0,556583.777778
19585,2021-11-30,Vente,20037016.0,1360.0,BELIGNEUX,01,32,Appartement,36.0,2.0,3000.0,556583.777778
19586,2021-11-30,Vente,20037016.0,1360.0,BELIGNEUX,01,32,Appartement,36.0,2.0,3000.0,556583.777778
19587,2021-11-30,Vente,20037016.0,1360.0,BELIGNEUX,01,32,Appartement,36.0,2.0,3000.0,556583.777778
19588,2021-11-30,Vente,20037016.0,1360.0,BELIGNEUX,01,32,Appartement,36.0,2.0,3000.0,556583.777778


In [87]:
df_clean = df_clean.drop_duplicates()

In [88]:
df_clean.duplicated().sum()

np.int64(0)

In [89]:
df_clean["prix_m2"] = (
    df_clean["Valeur fonciere"]
    / df_clean["Surface reelle bati"]
)

In [90]:
df_clean["prix_m2"].describe()

count    5.712235e+06
mean     5.987170e+03
std      4.295680e+04
min      3.960396e+00
25%      1.663889e+03
50%      2.619048e+03
75%      4.235294e+03
max      1.335833e+07
Name: prix_m2, dtype: float64

In [91]:
df_clean["prix_m2"].quantile([
    0.90,
    0.95,
    0.99,
    0.995,
    0.999
])

0.900      7500.000000
0.950     11341.152941
0.990     47378.933594
0.995    109189.672414
0.999    562172.450000
Name: prix_m2, dtype: float64

Une analyse des doublons a mis en évidence 277 963 observations dupliquées (4,64 % du jeu de données). Ces doublons correspondaient à des enregistrements strictement identiques et ont été supprimés avant les étapes d'analyse et de modélisation afin d'éviter un biais statistique.

In [92]:
df_clean.nlargest(
    30,
    "prix_m2"
)[[
    "Date mutation",
    "Commune",
    "Type local",
    "Valeur fonciere",
    "Surface reelle bati",
    "Nombre pieces principales",
    "prix_m2"
]]

,Date mutation,Commune,Type local,Valeur fonciere,Surface reelle bati,Nombre pieces principales,prix_m2
16639246,2024-04-23,PARIS 06,Appartement,80150000.0,6.0,1.0,1.335833e+07
9281255,2022-07-26,PARIS 08,Appartement,57500000.0,6.0,1.0,9.583333e+06
16639227,2024-04-23,PARIS 06,Appartement,80150000.0,10.0,1.0,8.015000e+06
4650670,2021-10-11,PARIS 16,Appartement,78000000.0,11.0,1.0,7.090909e+06
4598901,2021-05-31,PARIS 08,Appartement,56000000.0,8.0,1.0,7.000000e+06
13167137,2023-12-28,PARIS 13,Appartement,91000000.0,13.0,1.0,7.000000e+06
13167493,2023-12-28,PARIS 13,Appartement,91000000.0,13.0,2.0,7.000000e+06
16667250,2024-12-19,PARIS 16,Appartement,68820000.0,10.0,1.0,6.882000e+06
13167077,2023-12-28,PARIS 13,Appartement,91000000.0,14.0,1.0,6.500000e+06
8989752,2022-09-29,LEVALLOIS-PERRET,Appartement,84089544.0,13.0,1.0,6.468426e+06


In [93]:
(df_clean["prix_m2"] > 100_000).sum()

np.int64(30749)

In [94]:
(
    (df_clean["prix_m2"] > 100_000).sum()
    / len(df_clean)
) * 100

np.float64(0.5383006826574886)

In [95]:
df_ml = df_clean.copy()

In [96]:
df_ml = df_ml[
    df_ml["Surface reelle bati"] >= 15
].copy()

In [97]:
df_ml["prix_m2"] = (
    df_ml["Valeur fonciere"]
    / df_ml["Surface reelle bati"]
)

In [98]:
df_ml["prix_m2"].quantile([
    0.90,
    0.95,
    0.99,
    0.995,
    0.999
])

0.900      7352.941176
0.950     11041.666667
0.990     44117.808590
0.995    103333.333333
0.999    534322.367424
Name: prix_m2, dtype: float64

In [99]:
df_ml["annee"] = df_ml["Date mutation"].dt.year

df_ml["mois"] = df_ml["Date mutation"].dt.month

df_ml["trimestre"] = df_ml["Date mutation"].dt.quarter

In [100]:
df_ml[
    ["Date mutation", "annee", "mois", "trimestre"]
].head()

,Date mutation,annee,mois,trimestre
1,2021-01-05,2021,1,1
3,2021-01-04,2021,1,1
5,2021-01-06,2021,1,1
7,2021-01-04,2021,1,1
14,2021-01-04,2021,1,1


In [101]:
df_ml["annee"].value_counts().sort_index()

annee
2021    1372900
2022    1306309
2023    1026022
2024     929493
2025    1043674
Name: count, dtype: int64

In [102]:
df_ml.groupby(
    "annee"
)["prix_m2"].mean()

annee
2021    5377.316073
2022    6369.281650
2023    5360.667671
2024    5917.950161
2025    5869.527829
Name: prix_m2, dtype: float64

In [103]:
df_ml.groupby(
    "annee"
)["prix_m2"].median()

annee
2021    2474.000000
2022    2700.000000
2023    2677.014094
2024    2571.428571
2025    2638.888889
Name: prix_m2, dtype: float64

In [104]:
df_ml.groupby(
    "Type local"
)["prix_m2"].median()

Type local
Appartement    3500.000000
Maison         2162.162162
Name: prix_m2, dtype: float64

In [105]:
df_ml.groupby(
    "Code departement"
)["prix_m2"].median().sort_values(
    ascending=False
).head(20)

Code departement
75     10580.645161
92      7157.029491
94      5235.294118
06      4882.352941
74      4420.833333
2A      4321.428571
93      4291.666667
83      4108.846154
78      4066.393443
69      4000.000000
13      3787.419355
73      3600.000000
95      3578.313253
33      3563.138298
91      3368.220339
34      3251.470906
44      3205.882353
77      3175.301205
2B      3171.568627
971     3088.235294
Name: prix_m2, dtype: float64

In [106]:
import pandas as pd
import json

communes = pd.read_json("../../data/raw/geo/communes_france.json")

communes.head()

,nom,code,codeDepartement,codeRegion,population,centre
0,L'Abergement-Clémenciat,01001,01,84,860.0,"{'type': 'Point', 'coordinates': [4.9306, 46.1..."
1,L'Abergement-de-Varey,01002,01,84,270.0,"{'type': 'Point', 'coordinates': [5.4247, 46.0..."
2,Ambérieu-en-Bugey,01004,01,84,15934.0,"{'type': 'Point', 'coordinates': [5.3706, 45.9..."
3,Ambérieux-en-Dombes,01005,01,84,1906.0,"{'type': 'Point', 'coordinates': [4.9119, 45.9..."
4,Ambléon,01006,01,84,115.0,"{'type': 'Point', 'coordinates': [5.5928, 45.7..."


In [107]:
communes["longitude"] = communes["centre"].apply(
    lambda x: x["coordinates"][0] if isinstance(x, dict) else None
)

communes["latitude"] = communes["centre"].apply(
    lambda x: x["coordinates"][1] if isinstance(x, dict) else None
)

communes = communes.drop(columns=["centre"])


In [108]:
communes = communes.rename(
    columns={
        "code": "code_commune_geo",
        "nom": "nom_commune_geo",
        "population": "population_geo"
    }
)

In [109]:
communes.head()

,nom_commune_geo,code_commune_geo,codeDepartement,codeRegion,population_geo,longitude,latitude
0,L'Abergement-Clémenciat,01001,01,84,860.0,4.9306,46.1517
1,L'Abergement-de-Varey,01002,01,84,270.0,5.4247,46.0071
2,Ambérieu-en-Bugey,01004,01,84,15934.0,5.3706,45.9575
3,Ambérieux-en-Dombes,01005,01,84,1906.0,4.9119,45.9992
4,Ambléon,01006,01,84,115.0,5.5928,45.7483


In [120]:
def corriger_code_commune(row):
    dep = str(row["Code departement"]).zfill(2)
    code = str(row["Code commune"])

    # Paris arrondissements -> Paris
    if dep == "75":
        return "75056"

    # Marseille arrondissements -> Marseille
    if dep == "13" and code.zfill(3) in [str(i).zfill(3) for i in range(201, 217)]:
        return "13055"

    # Lyon arrondissements -> Lyon
    if dep == "69" and code.zfill(3) in [str(i).zfill(3) for i in range(381, 390)]:
        return "69123"

    # DOM : 971, 972, 973, 974, 976
    if len(dep) == 3:
        return dep + code.zfill(2)

    # Métropole
    return dep + code.zfill(3)

In [121]:
df_ml["code_commune_geo"] = df_ml.apply(
    corriger_code_commune,
    axis=1
)

df_final = df_ml.merge(
    communes,
    on="code_commune_geo",
    how="left"
)

In [122]:
df_final[["Commune", "nom_commune_geo", "latitude", "longitude", "population_geo"]].isnull().sum()

Commune               0
nom_commune_geo    7737
latitude           7737
longitude          7737
population_geo     7737
dtype: int64

In [123]:
df_final[df_final["nom_commune_geo"].isna()][
    ["Code departement", "Code commune", "Commune", "code_commune_geo"]
].drop_duplicates().head(30)

,Code departement,Code commune,Commune,code_commune_geo
2735,01,39,BEON,01039
4840,01,330,RUFFIEU,01330
13305,02,695,SAINT THIBAUT,02695
13735,02,77,BERZY LE SEC,02077
87962,08,294,LA MONCELLE,08294
90811,09,287,SENCONAC,09287
111975,12,76,CONQUES-EN-ROUERGUE,12076
157774,14,623,SAINT-MARTIN-DE-FONTENAY,14623
167350,14,11,AURSEULLES,14011
174738,14,300,GERROTS,14300


In [124]:
df_final[df_final["nom_commune_geo"].isna()][
    "Code departement"
].value_counts().head(20)

Code departement
971    2095
49     1140
93      982
69      713
85      463
14      323
62      222
16      218
22      203
72      196
12      114
79      103
61       97
25       93
71       67
19       52
42       45
86       45
01       43
88       40
Name: count, dtype: int64

In [125]:
df_final = df_final.dropna(
    subset=["latitude", "longitude", "population_geo"]
).copy()

In [126]:
df_final[["nom_commune_geo", "latitude", "longitude", "population_geo"]].isnull().sum()

nom_commune_geo    0
latitude           0
longitude          0
population_geo     0
dtype: int64

In [128]:
revenus = pd.read_csv(
    "../../data/raw/insee/revenus_communes.csv",
    sep=";"
)

revenus.head()

,CODGEO,LIBGEO,MED14
0,01001,L'Abergement-Clémenciat,"21576,7"
1,01002,L'Abergement-de-Varey,"21672,9"
2,01004,Ambérieu-en-Bugey,"19756,1"
3,01005,Ambérieux-en-Dombes,"23204,8"
4,01006,Ambléon,"22157,5"


In [129]:
revenus.columns

Index(['CODGEO', 'LIBGEO', 'MED14'], dtype='object')

In [130]:
revenus.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36621 entries, 0 to 36620
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   CODGEO  36621 non-null  object
 1   LIBGEO  36621 non-null  object
 2   MED14   32974 non-null  object
dtypes: object(3)
memory usage: 858.4+ KB


In [131]:
revenus["MED14"].head(20)

0     21576,7
1     21672,9
2     19756,1
3     23204,8
4     22157,5
5     21679,3
6     23839,0
7     21824,0
8     21962,0
9     21885,7
10    20885,7
11    20654,1
12    19806,0
13    22107,3
14    20138,1
15    18959,3
16        NaN
17    22680,4
18    18886,7
19        NaN
Name: MED14, dtype: object

In [132]:
revenus.sample(10)

,CODGEO,LIBGEO,MED14
11888,31013,Ardiège,"21409,7"
4068,11416,Villarzel-Cabardès,"16867,0"
2686,08286,Ménil-Annelles,"20560,8"
19908,51618,Le Vézier,"20715,2"
30510,74032,Bellevaux,"22162,0"
5165,14690,Tierceville,"20618,9"
10516,28036,Berchères-sur-Vesgre,"25488,8"
2931,09042,La Bastide-de-Sérou,"17633,0"
32508,79220,Prin-Deyrançon,"19131,2"
26790,64560,Viven,"21556,5"


In [133]:
df_final["code_commune_geo"].dtype

dtype('O')

In [134]:
df_final["code_commune_geo"].head()

0    01426
1    01065
2    01254
3    01344
4    01301
Name: code_commune_geo, dtype: object

In [135]:
revenus["MED14"] = (
    revenus["MED14"]
    .str.replace(",", ".", regex=False)
)

revenus["MED14"] = pd.to_numeric(
    revenus["MED14"],
    errors="coerce"
)

In [136]:
revenus = revenus.rename(
    columns={
        "CODGEO": "code_commune_geo",
        "LIBGEO": "nom_commune_revenu",
        "MED14": "revenu_median"
    }
)

In [137]:
df_final = df_final.merge(
    revenus[["code_commune_geo", "revenu_median"]],
    on="code_commune_geo",
    how="left"
)

In [138]:
df_final[["revenu_median"]].isnull().sum()

revenu_median    45573
dtype: int64

In [139]:
(df_final["revenu_median"].isnull().sum() / len(df_final)) * 100

np.float64(0.8036629239518991)

In [140]:
df_final = df_final.dropna(
    subset=["revenu_median"]
).copy()

In [141]:
df_final.isnull().sum().sort_values(ascending=False).head(20)

Surface terrain              2146437
Nature mutation                    0
Date mutation                      0
Code postal                        0
Commune                            0
Code departement                   0
Valeur fonciere                    0
Code commune                       0
Type local                         0
Surface reelle bati                0
Nombre pieces principales          0
prix_m2                            0
annee                              0
mois                               0
trimestre                          0
code_commune_geo                   0
nom_commune_geo                    0
codeDepartement                    0
codeRegion                         0
population_geo                     0
dtype: int64

In [142]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5625088 entries, 0 to 5670660
Data columns (total 23 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   Date mutation              datetime64[ns]
 1   Nature mutation            object        
 2   Valeur fonciere            float64       
 3   Code postal                float64       
 4   Commune                    object        
 5   Code departement           object        
 6   Code commune               int64         
 7   Type local                 object        
 8   Surface reelle bati        float64       
 9   Nombre pieces principales  float64       
 10  Surface terrain            float64       
 11  prix_m2                    float64       
 12  annee                      int32         
 13  mois                       int32         
 14  trimestre                  int32         
 15  code_commune_geo           object        
 16  nom_commune_geo            object        

In [143]:
population = pd.read_csv(
    "../../data/raw/insee/population_communes.csv",
    sep=";"
)

population.head()

,Code Officiel Région,Nom Officiel Région,Code Officiel Commune / Arrondissement Municipal,Nom Officiel Commune / Arrondissement Municipal,Population municipale,Population comptée à part,Population totale,Code Officiel Département,Code Officiel Arrondissement Départemental,Année de recensement,Année d’entrée en vigueur,Année de référence géographique,Nom Officiel EPCI,Code Officiel EPCI,Nom Officiel Département
0,84,Auvergne-Rhône-Alpes,38434,Saint-Ondras,645,6,651,38,2,2018,2021,2020,CC Les Vals du Dauphiné,200068567,Isère
1,84,Auvergne-Rhône-Alpes,38006,Allevard,4062,106,4168,38,1,2018,2021,2020,CC Le Grésivaudan,200018166,Isère
2,84,Auvergne-Rhône-Alpes,38259,Montseveroux,964,16,980,38,3,2018,2021,2020,CC d'Entre Bièvre et Rhône,200085751,Isère
3,84,Auvergne-Rhône-Alpes,38219,Marcollin,658,16,674,38,3,2018,2021,2020,CC Bièvre Isère,200059392,Isère
4,84,Auvergne-Rhône-Alpes,38463,Saint-Vérand,1716,38,1754,38,1,2018,2021,2020,CC Saint-Marcellin Vercors Isère Communauté,200070431,Isère


In [145]:
print(population.columns)

Index(['Code Officiel Région', 'Nom Officiel Région',
       'Code Officiel Commune / Arrondissement Municipal',
       'Nom Officiel Commune / Arrondissement Municipal',
       'Population municipale', 'Population comptée à part',
       'Population totale', 'Code Officiel Département',
       'Code Officiel Arrondissement Départemental', 'Année de recensement',
       'Année d’entrée en vigueur', 'Année de référence géographique',
       'Nom Officiel EPCI', 'Code Officiel EPCI', 'Nom Officiel Département'],
      dtype='object')


In [146]:
population_clean = population[
    [
        "Code Officiel Commune / Arrondissement Municipal",
        "Population municipale",
        "Population comptée à part",
        "Population totale",
        "Année de recensement",
        "Année d’entrée en vigueur",
        "Année de référence géographique"
    ]
].copy()

population_clean = population_clean.rename(
    columns={
        "Code Officiel Commune / Arrondissement Municipal": "code_commune_geo",
        "Population municipale": "population_municipale",
        "Population comptée à part": "population_comptee_a_part",
        "Population totale": "population_totale",
        "Année de recensement": "annee_recensement",
        "Année d’entrée en vigueur": "annee_entree_vigueur",
        "Année de référence géographique": "annee_reference_geo"
    }
)

In [147]:
population_clean.head()
population_clean.dtypes

code_commune_geo             int64
population_municipale        int64
population_comptee_a_part    int64
population_totale            int64
annee_recensement            int64
annee_entree_vigueur         int64
annee_reference_geo          int64
dtype: object

In [149]:
population_clean["code_commune_geo"] = (
    population_clean["code_commune_geo"]
    .astype(str)
    .str.zfill(5)
)

df_final["code_commune_geo"] = (
    df_final["code_commune_geo"]
    .astype(str)
    .str.zfill(5)
)

In [150]:
df_final = df_final.merge(
    population_clean,
    on="code_commune_geo",
    how="left"
)

In [151]:
df_final[
    ["population_geo", "population_municipale", "population_totale"]
].isnull().sum()

population_geo                 0
population_municipale    5524280
population_totale        5524280
dtype: int64

In [152]:
population_clean["code_commune_geo"].head(20)

0     38434
1     38006
2     38259
3     38219
4     38463
5     38509
6     38559
7     38335
8     38253
9     38549
10    38106
11    38314
12    38343
13    38276
14    38061
15    38207
16    38412
17    38351
18    38194
19    38439
Name: code_commune_geo, dtype: object

In [153]:
population_clean["code_commune_geo"].sample(20)

395    38264
440    38278
16     38412
444    38295
302    38174
220    38089
322    38255
178    38467
43     38023
368    38232
370    38515
445    38494
502    38039
325    38315
291    38186
13     38276
354    38065
66     38244
226    38126
27     38399
Name: code_commune_geo, dtype: object

In [154]:
df_final["code_commune_geo"].head(20)

0     01426
1     01065
2     01254
3     01344
4     01301
5     01289
6     01163
7     01163
8     01151
9     01370
10    01159
11    01053
12    01350
13    01289
14    01053
15    01289
16    01289
17    01053
18    01451
19    01264
Name: code_commune_geo, dtype: object

In [155]:
population_clean["code_commune_geo"].nunique()

512

In [156]:
df_final["code_commune_geo"].nunique()

29805

In [157]:
set(
    df_final["code_commune_geo"]
).intersection(
    set(population_clean["code_commune_geo"])
)

{'38001',
 '38002',
 '38003',
 '38004',
 '38005',
 '38006',
 '38009',
 '38010',
 '38011',
 '38012',
 '38013',
 '38015',
 '38017',
 '38018',
 '38019',
 '38020',
 '38022',
 '38023',
 '38026',
 '38027',
 '38029',
 '38030',
 '38032',
 '38033',
 '38034',
 '38035',
 '38037',
 '38038',
 '38039',
 '38040',
 '38041',
 '38042',
 '38043',
 '38044',
 '38045',
 '38046',
 '38047',
 '38048',
 '38049',
 '38050',
 '38051',
 '38052',
 '38053',
 '38054',
 '38055',
 '38057',
 '38058',
 '38059',
 '38060',
 '38061',
 '38062',
 '38063',
 '38064',
 '38065',
 '38066',
 '38067',
 '38068',
 '38069',
 '38070',
 '38071',
 '38072',
 '38074',
 '38075',
 '38076',
 '38077',
 '38078',
 '38080',
 '38081',
 '38082',
 '38083',
 '38084',
 '38085',
 '38086',
 '38087',
 '38089',
 '38090',
 '38091',
 '38093',
 '38094',
 '38095',
 '38097',
 '38098',
 '38099',
 '38100',
 '38101',
 '38102',
 '38103',
 '38104',
 '38105',
 '38106',
 '38107',
 '38108',
 '38109',
 '38110',
 '38111',
 '38113',
 '38114',
 '38115',
 '38117',
 '38118',
